# MuJoCo Robot Zoo — Interactive Demo

This notebook demonstrates how to launch physics simulations for robots from the
[MuJoCo Menagerie](https://github.com/google-deepmind/mujoco_menagerie) collection — a
curated library of high-quality robot models for the
[MuJoCo](https://mujoco.org/) physics engine.

**What does this notebook do:**
1. Open a virtual desktop (VNC) where the 3D viewer will appear.
2. Automatically discover every robot scene in `../mujoco_menagerie`.
3. Click a button to instantly launch any robot in an interactive MuJoCo viewer.

> **Tip — multiple scenes per robot:**  
> Some robots ship with several scene variants (e.g. `scene.xml`, `scene_mjx.xml`, `scene_arm.xml`).  
> Each variant gets its own button so you can try them all.

## Step 1 — Open the VNC Desktop

The MuJoCo viewer is a native OpenGL window.  
Run the cell below to embed the virtual desktop right here in JupyterLab.  
The simulation window will appear there once you launch a robot in Step 5.

> **VSCode users:**
> - When running a cell for the first time, VSCode will prompt you to select a Python environment — choose the conda base environment: `/opt/conda/bin/python`
> - The desktop will **not** pop up automatically. Click the blue button below to open it in a new browser tab.

In [ ]:
from utils import display_desktop
display_desktop()

## Step 2 — Import Libraries

| Library | Purpose |
|---|---|
| `mujoco` | Physics engine & MJCF model loader |
| `mujoco.viewer` | Passive (non-blocking) 3-D viewer |
| `threading` | Run the simulation loop in the background |
| `ipywidgets` | Interactive buttons and layout |
| `glob` | Discover XML scene files on disk |

In [ ]:
import os
import time
import threading
import glob as glob_module

import numpy as np
import mujoco
import mujoco.viewer
import ipywidgets as widgets
from IPython.display import display

print(f"MuJoCo version: {mujoco.__version__}")

## Step 3 — Simulation Engine

The `policy` function is called every simulation step and writes actuator commands into `d.ctrl`.  
It currently applies **random controls** as a placeholder — replace it with your own control logic.

`run_sim` is the physics loop running inside a background thread:

1. **Load model** — `MjModel.from_xml_path()` parses the MJCF scene file.
2. **Create data** — `MjData` holds the mutable simulation state (positions, velocities, forces …).
3. **Passive viewer** — `launch_passive` opens the viewer without blocking Python; the notebook stays responsive.
4. **Policy** — `policy(m, d)` is called each step to compute and apply actuator commands.
5. **Real-time pacing** — the loop sleeps for exactly `model.opt.timestep` seconds between steps.

`launch_sim` wraps `run_sim`: it stops any already-running simulation before starting the new one,
so clicking a second robot automatically closes the first viewer.

In [ ]:
# ── Global simulation state ──────────────────────────────────────────────────
_sim_stop_event: threading.Event = threading.Event()
_sim_thread: threading.Thread | None = None
_policy_enabled: bool = False


def policy(m: mujoco.MjModel, d: mujoco.MjData) -> None:
    """Placeholder policy — replace with your own control logic.

    Args:
        m: MuJoCo model (read-only simulation parameters).
        d: MuJoCo data (current simulation state); write d.ctrl[:] to apply commands.
    """
    if not _policy_enabled:
        return
    if m.nu > 0:
        ctrl_range = m.actuator_ctrlrange
        d.ctrl[:] = np.random.uniform(
            low=ctrl_range[:, 0],
            high=ctrl_range[:, 1],
        )


def run_sim(xml_path: str, stop_event: threading.Event) -> None:
    """Load an MJCF scene and run a real-time physics loop in a passive viewer."""
    m = mujoco.MjModel.from_xml_path(xml_path)
    d = mujoco.MjData(m)

    with mujoco.viewer.launch_passive(m, d, show_left_ui=False) as viewer:
        # Disable shadows and reflections for better performance
        viewer.user_scn.flags[mujoco.mjtRndFlag.mjRND_SHADOW] = 0
        viewer.user_scn.flags[mujoco.mjtRndFlag.mjRND_REFLECTION] = 0

        start = time.time()
        # Run for up to 10 minutes, or until the viewer is closed / stop is requested
        while viewer.is_running() and not stop_event.is_set() and time.time() - start < 600:
            step_start = time.time()

            policy(m, d)

            mujoco.mj_step(m, d)
            viewer.sync()

            # Real-time pacing: sleep for the remainder of the timestep
            elapsed = time.time() - step_start
            remaining = m.opt.timestep - elapsed
            if remaining > 0:
                time.sleep(remaining)


def launch_sim(xml_path: str) -> None:
    """Stop any running simulation and start a new one for *xml_path*."""
    global _sim_stop_event, _sim_thread

    # Signal the previous thread to stop and wait briefly for it to exit
    _sim_stop_event.set()
    if _sim_thread is not None and _sim_thread.is_alive():
        _sim_thread.join(timeout=2.0)

    # Start fresh
    _sim_stop_event = threading.Event()
    _sim_thread = threading.Thread(
        target=run_sim,
        args=(xml_path, _sim_stop_event),
        daemon=True,
    )
    _sim_thread.start()


print("Simulation engine ready.")

## Step 4 — Discover Robot Scenes

The cell below scans every sub-directory of `../mujoco_menagerie` and collects:

- All `scene*.xml` files (the MJCF scene descriptions).
- The first `.png` / `.jpg` image found in that directory, compressed to a small JPEG thumbnail for fast rendering.

Directories that contain no `scene*.xml` files (documentation, assets, tests …) are skipped.

In [ ]:
import base64
import io
from PIL import Image as PILImage

MENAGERIE_DIR = os.path.normpath(os.path.join(os.getcwd(), "..", "mujoco_menagerie"))

robots: list[dict] = []

for entry in sorted(os.listdir(MENAGERIE_DIR)):
    robot_path = os.path.join(MENAGERIE_DIR, entry)
    if not os.path.isdir(robot_path):
        continue

    scene_files = sorted(glob_module.glob(os.path.join(robot_path, "scene*.xml")))
    if not scene_files:
        continue

    images = (glob_module.glob(os.path.join(robot_path, "*.png")) +
              glob_module.glob(os.path.join(robot_path, "*.jpg")) +
              glob_module.glob(os.path.join(robot_path, "*.jpeg")))
    img_path = images[0] if images else None

    robots.append({
        "name": entry,
        "path": robot_path,
        "scenes": scene_files,
        "image": img_path,
        "thumbnail": None,
    })


def _make_thumbnail(path: str, w: int = 164, h: int = 100) -> str | None:
    """Return a base64 JPEG data-URI scaled to (w x h), or None on error."""
    try:
        with PILImage.open(path) as img:
            img = img.convert("RGB")
            img.thumbnail((w, h), PILImage.LANCZOS)
            buf = io.BytesIO()
            img.save(buf, format="JPEG", quality=75)
            return "data:image/jpeg;base64," + base64.b64encode(buf.getvalue()).decode()
    except Exception:
        return None


for robot in robots:
    robot["thumbnail"] = _make_thumbnail(robot["image"]) if robot["image"] else None

print(f"Found {len(robots)} robots\n")

## Step 5 — Interactive Robot Launcher

Run the cell below to display the robot gallery.

- Each card shows the **robot thumbnail** (if available) and one **button per scene variant**.
- Click any button to load and launch that robot in the VNC viewer (Step 1).
- Launching a new robot automatically **closes the previous viewer** — no need to restart the kernel.
- Use the **Policy** toggle to enable or disable the policy function at any time:
  - **ON** — `policy(m, d)` is called every simulation step (default: random controls)
  - **OFF** — no control commands are sent; the robot remains in its resting pose. Useful when you want to manually control joints via the **Control** panel in the viewer

> **Controls inside the viewer:**  
> - Expand **Joint** to inspect current joint states
> - Expand **Control** to send control commands to the robot.
> - `Left-drag` — rotate · `Right-drag` — pan · `Scroll` — zoom · `Ctrl+A` — reset camera  
> - Close the window to stop the current simulation.

In [ ]:
CARD_WIDTH   = 120
COLS_PER_ROW = 5

# ── Shared card styles ───────────────────────────────────────────────────────
display(widgets.HTML("""<style>
.robot-card img { width: 164px; height: 100px; object-fit: contain; display: block; }
.robot-card .name { font-size: 12px; color: #333; word-break: break-word;
                    text-align: center; padding: 2px 0 4px; }
</style>"""))

# ── Toolbar: status bar + policy toggle ──────────────────────────────────────
status_bar = widgets.HTML("<b>Click a button below to launch a simulation.</b>")

policy_toggle = widgets.ToggleButton(
    value=_policy_enabled,
    description="Policy: ON" if _policy_enabled else "Policy: OFF",
    button_style="success" if _policy_enabled else "warning",
    tooltip="Toggle whether the policy sends control commands each simulation step",
    layout=widgets.Layout(width="140px", height="32px"),
)

def _on_policy_toggle(change):
    global _policy_enabled
    _policy_enabled = change["new"]
    policy_toggle.description = "Policy: ON" if change["new"] else "Policy: OFF"
    policy_toggle.button_style = "success" if change["new"] else "warning"

policy_toggle.observe(_on_policy_toggle, names="value")


def make_card(robot: dict) -> widgets.VBox:
    """Build one robot card: thumbnail + one launch button per scene."""
    if robot["thumbnail"]:
        thumb_html = f'<img src="{robot["thumbnail"]}" alt="{robot["name"]}">'
    else:
        thumb_html = '<div style="height:100px;font-size:28px;display:flex;align-items:center;justify-content:center">🤖</div>'

    header = widgets.HTML(
        f'<div class="robot-card">{thumb_html}<div class="name">{robot["name"]}</div></div>'
    )

    buttons = []
    for scene_path in robot["scenes"]:
        scene_name = os.path.basename(scene_path).replace(".xml", "")
        btn = widgets.Button(
            description=scene_name,
            tooltip=f"Launch {robot['name']} — {scene_name}",
            button_style="primary",
            layout=widgets.Layout(width="164px", height="26px", margin="1px 0"),
        )

        def _on_click(b, path=scene_path, rname=robot["name"], sname=scene_name):
            status_bar.value = f"⏳ Launching <b>{rname}</b> — <code>{sname}</code> …"
            launch_sim(path)
            status_bar.value = (
                f"✅ Running <b>{rname}</b> — <code>{sname}</code>. "
                "Switch to the VNC Desktop tab to see the viewer."
            )

        btn.on_click(_on_click)
        buttons.append(btn)

    return widgets.VBox(
        [header] + buttons,
        layout=widgets.Layout(
            overflow="hidden",
            border="1px solid #e0e0e0", border_radius="6px",
            padding="8px", width=f"{CARD_WIDTH}px", align_items="center",
        ),
    )


# ── Build gallery ────────────────────────────────────────────────────────────
cards = [make_card(r) for r in robots]

gallery = widgets.GridBox(
    cards,
    layout=widgets.Layout(
        grid_template_columns=f"repeat({COLS_PER_ROW}, {CARD_WIDTH + 12}px)",
    ),
)

display(widgets.VBox([policy_toggle, status_bar, gallery]))